# GaSi-Labor: Dateisichtung

Dieses Notebook zeigt die Ablageorte und prÃ¼ft, welche Dateien dort liegen.

## Ablagepfade

In [ ]:
from pathlib import Path

repo_root = Path.cwd()
folders = {
    "GerÃ¤teunterlagen": repo_root / "data" / "datasheets",
    "Messdaten": repo_root / "data" / "measurements",
    "Fachliteratur": repo_root / "data" / "paper",
    "Software-Dokumentation": repo_root / "data" / "wiki",
    "Code": repo_root / "code",
}

print("Ausgewertete Ablagepfade (relativ zum Repository):")
for label, folder in folders.items():
    print(f"- {label}: {folder.relative_to(repo_root).as_posix()}/")

## Dateien je Ordner

In [ ]:
empty_instructions = {
    "GerÃ¤teunterlagen": "Bitte DatenblÃ¤tter, HandbÃ¼cher und weitere GerÃ¤teunterlagen in data/datasheets/ ablegen.",
    "Messdaten": "Bitte Messdaten in data/measurements/ ablegen.",
    "Fachliteratur": "Bitte VerÃ¶ffentlichungen und Fachartikel in data/paper/ ablegen.",
    "Software-Dokumentation": "Bitte Wiki- und Software-Dokumentation in data/wiki/ ablegen.",
    "Code": "Bitte relevante Quellcode-Dateien oder VerknÃ¼pfungen in code/ ablegen.",
}

file_lists = {}
for label, folder in folders.items():
    files = sorted(
        path for path in folder.rglob("*")
        if path.is_file()
        and path.name not in {".gitkeep", "_local_openbis_import_preview.json"}
    )
    file_lists[label] = files
    print(f"\n{label} ({folder.relative_to(repo_root).as_posix()}/):")
    if files:
        for path in files:
            print(f"- {path.relative_to(repo_root).as_posix()}")
    else:
        print(f"Keine Dateien gefunden. {empty_instructions[label]}")

## Lokale PrÃ¼fung und Importvorschau

Die Zellen unten lesen und validieren die CSVs nur lokal. Es gibt keine openBIS-Verbindung. Drei Beispielpunkte liegen ausschlieÃŸlich in der ignorierten lokalen Vorschau; Notebook-Ausgaben zeigen keine Messpunktwerte.

In [ ]:
import csv
import hashlib
import json
from collections import Counter
from datetime import datetime
from pathlib import Path

repo_root = Path.cwd()
measurement_dir = repo_root / "data" / "measurements"
csv_paths = sorted(path for path in measurement_dir.glob("*.csv") if path.is_file())
if not csv_paths:
    raise FileNotFoundError("Keine CSV-MesslÃ¤ufe in data/measurements/ gefunden.")


def number_or_none(raw):
    if raw is None or raw.strip() == "":
        return None
    try:
        return float(raw.replace(",", "."))
    except ValueError:
        return None


summaries = []
headers_seen = set()
project_names = set()
gas_profiles = set()
component_ids = set()
sampling_intervals = Counter()
ambiguous_clock_rollbacks = []
parse_errors = 0
row_shape_errors = 0
numeric_errors = 0
calibration_entries = 0
calibration_payloads = []
column_count = None
blank_columns = []
numeric_fields = (
    "K1_KONZENTRATION", "K2_KONZENTRATION", "K3_KONZENTRATION",
    "MFC1_SOLL", "MFC1_IST", "MBW373_FROST_POINT_C",
    "MBW473SP2_FROST_POINT_C", "KLIKA_C", "PT_PROZESS_C",
    "A5B_CH01_mV", "KELLER_P1_bar",
)

for path in csv_paths:
    file_rows = 0
    start_time = None
    end_time = None
    previous_time = None
    file_intervals = Counter()
    file_projects = set()
    file_profiles = set()
    file_components = set()
    file_mbw373_online = False

    with path.open("r", encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source, delimiter="\t")
        headers = tuple(reader.fieldnames or ())
        headers_seen.add(headers)
        column_count = len(headers) if column_count is None else column_count
        blank_columns = [index for index, name in enumerate(headers) if name == ""]

        for row_number, row in enumerate(reader, start=2):
            if row.get("Datum") == "Datum":
                continue
            if None in row and row[None]:
                row_shape_errors += 1
            file_rows += 1

            try:
                timestamp = datetime.strptime(
                    f"{row.get('Datum', '')} {row.get('Uhrzeit', '')}",
                    "%d.%m.%Y %H:%M:%S",
                )
            except ValueError:
                parse_errors += 1
                continue

            if start_time is None:
                start_time = timestamp
            if previous_time is not None:
                interval = int((timestamp - previous_time).total_seconds())
                if interval < 0:
                    backward = abs(interval)
                    if timestamp.date() == previous_time.date() and 1800 <= backward <= 3600:
                        ambiguous_clock_rollbacks.append(
                            {"file": path.name, "line": row_number, "backward_seconds": backward}
                        )
                    else:
                        parse_errors += 1
                elif 0 < interval <= 120:
                    file_intervals[interval] += 1
                    sampling_intervals[interval] += 1
            previous_time = timestamp
            end_time = timestamp

            project = (row.get("PROJEKT_NAME") or "").strip()
            profile = (row.get("GASPROFIL_ID") or "").strip()
            if project:
                file_projects.add(project)
                project_names.add(project)
            if profile:
                file_profiles.add(profile)
                gas_profiles.add(profile)
            for slot in range(1, 4):
                gas_id = (row.get(f"K{slot}_GAS_ID") or "").strip()
                if gas_id:
                    file_components.add(gas_id)
                    component_ids.add(gas_id)
            file_mbw373_online |= number_or_none(row.get("MBW373_ONLINE")) == 1

            calibration_raw = (row.get("KALIBRIERUNGEN_JSON") or "").strip()
            if calibration_raw:
                calibration_payloads.append(calibration_raw)
                try:
                    calibration_entries += len(json.loads(calibration_raw))
                except (json.JSONDecodeError, TypeError):
                    parse_errors += 1

            for field in numeric_fields:
                raw_value = row.get(field)
                if raw_value and number_or_none(raw_value) is None:
                    numeric_errors += 1

    summaries.append(
        {
            "path": path,
            "rows": file_rows,
            "start": start_time,
            "end": end_time,
            "intervals": file_intervals,
            "projects": file_projects,
            "profiles": file_profiles,
            "components": file_components,
            "mbw373_online": file_mbw373_online,
        }
    )

assert len(headers_seen) == 1, f"Unterschiedliche CSV-Schemata gefunden: {len(headers_seen)}"
assert column_count == 252 and blank_columns == [2], "Unerwartete CSV-Kopfzeile"
assert parse_errors == 0 and numeric_errors == 0 and row_shape_errors == 0, "CSV-Validierung fehlgeschlagen"
assert all(item["rows"] for item in summaries), "Mindestens eine CSV enthÃ¤lt keine Datenzeilen."

example = next((item for item in summaries if item["components"] and item["mbw373_online"]), summaries[0])
example_path = example["path"]
point_indexes = {0, example["rows"] // 2, example["rows"] - 1}
point_fields = (
    "AUTOMATIK_AKTIV", "AUTOMATIK_STATUS", "GASPROFIL_ID",
    "K1_GAS_ID", "K1_GAS_NAME_DE", "K1_KONZENTRATION", "K1_EINHEIT",
    "MBW373_ONLINE", "MBW373_ID", "MBW373_FROST_POINT_C",
    "MBW373_HEAD_PRESSURE_PA", "MBW373_EXTERNAL_TEMP_C", "MBW373_FLOW_L_MIN",
    "MBW373_STABLE", "KLIKA_C", "PT_PROZESS_C", "KELLER_T1_C",
    "KELLER_P1_bar", "A5B_CH01_mV",
)
point_samples = []
with example_path.open("r", encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source, delimiter="\t")
    for index, row in enumerate(reader):
        if index not in point_indexes or row.get("Datum") == "Datum":
            continue
        point_samples.append(
            {
                "source_row_number": index + 2,
                "timestamp_local_no_timezone": f"{row.get('Datum', '')} {row.get('Uhrzeit', '')}",
                "raw_source_fields": {field: row.get(field) or None for field in point_fields},
            }
        )

example_project = sorted(example["projects"])[0] if example["projects"] else None
example_start = example["start"].isoformat(timespec="seconds") if example["start"] else None
example_end = example["end"].isoformat(timespec="seconds") if example["end"] else None
reference_id = next(
    (point["raw_source_fields"]["MBW373_ID"] for point in point_samples
     if point["raw_source_fields"]["MBW373_ID"]),
    None,
)
preview_path = measurement_dir / "_local_openbis_import_preview.json"
preview = {
    "preview_only": True,
    "openbis_write_performed": False,
    "source_run_key": {"value": f"LEGACY_FILE:{example_path.name}", "status": "file surrogate, not a session ID"},
    "source_file": {
        "path": example_path.relative_to(repo_root).as_posix(),
        "sha256": hashlib.sha256(example_path.read_bytes()).hexdigest(),
        "delimiter": "tab",
        "decimal_separator": ",",
        "columns": column_count,
        "rows": example["rows"],
        "start_local_no_timezone": example_start,
        "end_local_no_timezone": example_end,
        "sampling_intervals_seconds": dict(example["intervals"]),
    },
    "openbis_path_proposal": {
        "space": "<existing FB 8.1 Lab Notebook Space; verify code>",
        "project": "GASI",
        "collection": "MEASUREMENT_RUNS_2026",
        "object_type": "EXPERIMENTAL_STEP",
        "object_code": "server generated; no openBIS object created",
    },
    "object_properties": {
        "name_from_PROJEKT_NAME": example_project,
        "start_date_from_Datum_Uhrzeit": example_start,
        "end_date_from_Datum_Uhrzeit": example_end,
        "gas_profile_from_GASPROFIL_ID": sorted(example["profiles"]),
        "measurement_purpose": None,
        "dut_identifier": None,
    },
    "relationships": [
        {
            "direction": "INSTRUMENT parent -> EXPERIMENTAL_STEP",
            "candidate_reference_identifier": reference_id,
            "status": "resolve to existing Inventory Object in Playground",
        }
    ],
    "datasets": [
        {"type": "RAW_DATA", "file": example_path.relative_to(repo_root).as_posix()},
    ],
    "unresolved": [
        "DUT-to-A5B-channel mapping",
        "historical recipe/configuration snapshot",
        "calibration report and analyzed result file",
    ],
    "point_samples_saved_locally_only": point_samples,
}
preview_path.write_text(json.dumps(preview, ensure_ascii=False, indent=2), encoding="utf-8")

all_rows = sum(item["rows"] for item in summaries)
calibrations_empty = all(value == "[]" for value in calibration_payloads)
print("Lokale CSV-Validierung:")
print(f"- Dateien/Zeilen: {len(summaries)} / {all_rows}; Schema-Varianten/Spalten: {len(headers_seen)} / {column_count}")
print(f"- Parse-/Zahlen-/Zeilenfehler: {parse_errors}/{numeric_errors}/{row_shape_errors}")
print(f"- MÃ¶gliche lokale Uhr-RÃ¼ckstellungen: {len(ambiguous_clock_rollbacks)}; Zeitzone fehlt im CSV")
print(f"- HÃ¤ufige Abtastintervalle (s): {dict(sampling_intervals.most_common(5))}")
print(f"- Projektlabels/Gasprofile/Komponentengase: {len(project_names)}/{len(gas_profiles)}/{sorted(component_ids)}")
print(f"- KalibrierungseintrÃ¤ge: {calibration_entries}; JSON-Payloads leer: {calibrations_empty}")
print(f"- Beispiel-Datei: {example_path.name}; lokale Vorschau: {preview_path.relative_to(repo_root).as_posix()}")
print("- Beispiel-Messwerte werden nicht in Notebook-Ausgaben angezeigt.")
print("\nopenBIS-Importvorschau (nur Struktur):")
print(json.dumps(
    {
        "object_type": preview["openbis_path_proposal"]["object_type"],
        "property_sources": list(preview["object_properties"]),
        "relationships": preview["relationships"],
        "datasets": preview["datasets"],
        "unresolved": preview["unresolved"],
        "locally_saved_point_examples": len(point_samples),
    },
    ensure_ascii=False,
    indent=2,
))